# Sortino Momentum — Single-Asset Grid Search

Interactive validation of the Sortino-momentum pipeline that powers the
Momentum Grid frontend page.

**What this covers:**
- Load price data for a single symbol (e.g. GLD)
- Run the Sortino grid search (`analyze_momentum_grid_search`)
- Bootstrap significance test
- Current regime detection
- Inline heatmap and histogram plots

In [ ]:
import sys
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display

warnings.filterwarnings("ignore")
plt.style.use("seaborn-v0_8-darkgrid")
sns.set_palette("husl")
%matplotlib inline

pd.set_option("display.precision", 4)

CWD = Path.cwd().resolve()
for root in [CWD, CWD.parent]:
    if (root / "config" / "settings.py").exists():
        if str(root) not in sys.path:
            sys.path.insert(0, str(root))
        PROJECT_ROOT = root
        break
else:
    raise FileNotFoundError("Cannot find project root (config/settings.py missing)")

DATA_DIR = PROJECT_ROOT / "data" / "factors"
assert DATA_DIR.exists(), f"Data directory missing: {DATA_DIR}"
print(f"Project root : {PROJECT_ROOT}")
print(f"Data directory: {DATA_DIR}")

## 1. Load prices and pick a symbol

In [ ]:
prices = pd.read_parquet(DATA_DIR / "prices.parquet")

SYMBOL = "GLD"
if SYMBOL not in prices.columns:
    SYMBOL = prices.columns[0]
    print(f"GLD not found, falling back to {SYMBOL}")

price_series = prices[SYMBOL].dropna()
returns = price_series.pct_change().dropna()

print(f"Symbol: {SYMBOL}")
print(f"Date range: {returns.index[0].date()} .. {returns.index[-1].date()}")
print(f"Trading days: {len(returns):,}")

## 2. Grid search: optimal (X, K) by hit rate

In [ ]:
from core.signals.momentum import analyze_momentum_grid_search

grid_df = analyze_momentum_grid_search(returns, sortino_window=252, min_signals=10)
print(f"Grid combinations with enough signals: {len(grid_df)}")
display(grid_df.head(10))

In [ ]:
if not grid_df.empty:
    pivot = grid_df.pivot(
        index="X (lookback)", columns="K (forecast)", values="Z (hit_rate)"
    )

    fig, ax = plt.subplots(figsize=(8, 6))
    sns.heatmap(
        pivot, annot=True, fmt=".1f", cmap="RdYlGn",
        center=50, ax=ax, linewidths=0.5,
    )
    ax.set_title(f"{SYMBOL} — Sortino Momentum Hit Rate (%)")
    ax.set_xlabel("Forecast horizon K (days)")
    ax.set_ylabel("Lookback X (days)")
    plt.tight_layout()
    plt.show()
else:
    print("No valid grid combinations — not enough data or signals.")

## 3. Bootstrap significance test

In [ ]:
from core.signals.momentum import bootstrap_significance_test

best_x = int(grid_df.iloc[0]["X (lookback)"]) if not grid_df.empty else 10
best_k = int(grid_df.iloc[0]["K (forecast)"]) if not grid_df.empty else 10
print(f"Testing best (X={best_x}, K={best_k}) from grid search")

boot = bootstrap_significance_test(returns, x=best_x, k=best_k, sortino_window=252)

print(f"  Actual hit rate:  {boot.get('actual_hit_rate', 'N/A'):.1f}%")
print(f"  Random mean:      {boot.get('random_mean', 'N/A'):.1f}%")
print(f"  p-value:          {boot.get('p_value', 'N/A'):.4f}")
print(f"  Significant:      {boot.get('significant', 'N/A')}")
print(f"  # signals:        {boot.get('n_signals', 'N/A')}")

In [ ]:
dist = boot.get("bootstrap_dist", [])
actual = boot.get("actual_hit_rate")
rand_mean = boot.get("random_mean")

if dist and actual is not None:
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.hist(dist, bins=40, alpha=0.6, edgecolor="black", linewidth=0.5)
    ax.axvline(actual, color="green", linewidth=2, label=f"Actual: {actual:.1f}%")
    if rand_mean is not None:
        ax.axvline(rand_mean, color="red", linewidth=2, linestyle="--", label=f"Random mean: {rand_mean:.1f}%")
    ax.set_xlabel("Hit Rate (%)")
    ax.set_ylabel("Frequency")
    ax.set_title(f"{SYMBOL} — Bootstrap Distribution (X={best_x}, K={best_k})")
    ax.legend()
    plt.tight_layout()
    plt.show()
else:
    print("Not enough data for bootstrap plot.")

## 4. Current regime

In [ ]:
from core.signals.momentum import get_current_regime

regime = get_current_regime(returns, x=best_x, k=best_k, sortino_window=252)

if regime:
    print(f"Current Sortino:    {regime['current_sortino']:.3f}")
    print(f"Recent slope:       {regime['recent_slope']:.6f}")
    print(f"Baseline slope:     {regime['baseline_slope']:.6f}")
    print(f"Strong momentum:    {regime['strong_momentum']}")
else:
    print("Insufficient data for regime detection.")

## 5. Rolling Sortino over time

In [ ]:
from core.backtest.portfolio import calculate_rolling_metrics

rolling = calculate_rolling_metrics(returns, window=252)

fig, axes = plt.subplots(2, 1, figsize=(14, 6), sharex=True)

axes[0].plot(rolling.index, rolling["sortino_ratio"], linewidth=0.8)
axes[0].set_ylabel("Rolling Sortino")
axes[0].set_title(f"{SYMBOL} — Rolling 252-day Sortino")
axes[0].axhline(0, color="gray", linewidth=0.5)

axes[1].plot(rolling.index, rolling["sharpe_ratio"], linewidth=0.8, color="orange")
axes[1].set_ylabel("Rolling Sharpe")
axes[1].set_title(f"{SYMBOL} — Rolling 252-day Sharpe")
axes[1].axhline(0, color="gray", linewidth=0.5)

fig.tight_layout()
plt.show()